# DLA & Circuit Analysis

This notebook demonstrates InterpKit's circuit-level interpretability tools:

- **Direct Logit Attribution (DLA)** — decompose output logits by component
- **Head activations** — per-head output decomposition
- **Residual stream decomposition** — per-component norms in the residual stream
- **OV / QK analysis** — effective weight matrices per head
- **Composition scores** — how heads compose across layers
- **Circuit discovery** — automated minimal circuit finding

In [ ]:
import interpkit

model = interpkit.load("gpt2")

## Direct Logit Attribution (DLA)

DLA answers: *why does the model predict this token?* It projects each component's
output through the unembedding matrix and reports the logit contribution.

Results include both component-level (attn + MLP per layer) and per-head breakdowns.

In [ ]:
result = model.dla("The capital of France is")

print(f"Target token: {result['target_token']!r}")
print(f"Total logit:  {result['total_logit']:.3f}")
print("\nTop 5 component contributors:")
for c in result["contributions"][:5]:
    print(f"  {c['component']:12s}  {c['logit_contribution']:+.3f}")

print("\nTop 5 individual heads:")
for h in result["head_contributions"][:5]:
    print(f"  {h['component']:12s}  {h['logit_contribution']:+.3f}")

### DLA for a specific token

You can target a specific token instead of the model's top prediction.

In [ ]:
result = model.dla("The capital of France is", token="Paris", save="dla_paris.png")
print(f"Target: {result['target_token']!r}")
print(f"Top contributor: {result['contributions'][0]['component']} "
      f"({result['contributions'][0]['logit_contribution']:+.3f})")

## Head Activations

Decompose an attention module's output into per-head contributions.
With `output_proj=True` (default), each head's output is projected through
its slice of W_O, so the result lives in residual-stream space.

In [ ]:
heads = model.head_activations("The capital of France is", at="transformer.h.8")

print(f"Number of heads: {heads['num_heads']}")
print(f"Head dim:        {heads['head_dim']}")
print(f"Output shape:    {heads['head_acts'].shape}")
print("  → (num_heads, batch, seq, d_model)")

# Per-head norms
for h in range(heads['num_heads']):
    norm = heads['head_acts'][h].float().norm().item()
    print(f"  Head {h:2d}: norm = {norm:.2f}")

## Residual Stream Decomposition

Break down the residual stream at a given position into contributions from
each attention block and MLP. The norm of each contribution tells you how
much that component "writes" to the residual stream.

In [ ]:
decomp = model.decompose("The capital of France is")

print(f"Position analysed: {decomp['position']}")
print(f"Residual norm:     {decomp['residual'].norm().item():.2f}")
print("\nComponent norms:")
for comp in decomp["components"]:
    bar = '█' * int(comp['norm'] / 2)
    print(f"  {comp['name']:12s}  {comp['norm']:6.2f}  {bar}")

## OV Circuit Analysis

The OV matrix (W_O @ W_V) determines what an attention head writes to the residual
stream given the value it reads. Analysing its singular values and rank tells you
whether the head computes a low-rank or high-rank transformation.

In [ ]:
ov = model.ov_scores(layer=8)

print(f"\nLayer {ov['layer']} OV analysis:")
for h in ov["heads"]:
    svs = ', '.join(f'{v:.1f}' for v in h['top_singular_values'][:3])
    print(f"  Head {h['head']:2d}: rank≈{h['approx_rank']:3d}  "
          f"‖W_OV‖={h['frobenius_norm']:.1f}  top SVs=[{svs}]")

## QK Circuit Analysis

The QK matrix (W_Q^T @ W_K) determines what each head attends to.
Low-rank QK circuits often correspond to interpretable attention patterns.

In [ ]:
qk = model.qk_scores(layer=8)

print(f"\nLayer {qk['layer']} QK analysis:")
for h in qk["heads"]:
    svs = ', '.join(f'{v:.1f}' for v in h['top_singular_values'][:3])
    print(f"  Head {h['head']:2d}: rank≈{h['approx_rank']:3d}  "
          f"‖W_QK‖={h['frobenius_norm']:.1f}  top SVs=[{svs}]")

## Composition Scores

Measure how much heads in different layers compose with each other.
High Q-composition from L4.H3 → L8.H5 means head 3 in layer 4
writes something that head 5 in layer 8 attends to.

In [ ]:
comp = model.composition(src_layer=4, dst_layer=8, comp_type="q")

# The scores tensor is (dst_heads x src_heads)
print(f"Score matrix shape: {comp['scores'].shape}")
print("Top Q-composition pairs:")

scores = comp["scores"]
flat = scores.view(-1)
top_vals, top_idxs = flat.topk(5)
for val, idx in zip(top_vals.tolist(), top_idxs.tolist()):
    i = idx // scores.shape[1]
    j = idx % scores.shape[1]
    print(f"  L4.H{j} → L8.H{i}: {val:.4f}")

## Circuit Discovery

Automatically find the minimal set of components (attention blocks and MLPs)
that explain why the model outputs differently on two inputs.

The algorithm ablates each component individually, keeps those above a threshold,
and verifies the circuit by ablating all non-circuit components simultaneously.

In [ ]:
circuit = model.find_circuit(
    "The Eiffel Tower is in Paris",
    "The Eiffel Tower is in Rome",
    threshold=0.05,
)

print(f"Circuit size: {len(circuit['circuit'])} / {circuit['total_components']} components")
print(f"Faithfulness:  {circuit['verification']['faithfulness']:.1%}")
print("\nCircuit components:")
for c in circuit["circuit"][:10]:
    print(f"  {c['component']:12s} ({c['type']})  effect={c['effect']:.3f}")